In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
import pickle
from model.autoencoder import Model
from train import Trainer
from data import get_cross_data, get_rec_data, load_data

In [ ]:
unpaired_batch_size = 32
paired_batch_size = int(unpaired_batch_size/4)
T = 64          # Number of frames (64)
M = 1           # Number of persons
V = 25          # Number of joints
setting = 'cs'  # 'cs' or 'cv'
dataset = 'ntu120'

In [ ]:
# Load data
X = load_data(dataset)

# Try to load paired data from pickle file
if os.path.exists(f'data/{dataset}_{setting}_paired.pkl'):
    with open(f'data/{dataset}_{setting}_paired.pkl', 'rb') as f:
        paired_data = pickle.load(f)
        paired_train = paired_data['train']
        paired_test = paired_data['test']
        print('Paired data loaded from pickle file')
else:
    # Generate paired data and save to pickle file
    paired_train, paired_test = get_cross_data(X, dataset, setting, paired_batch_size, T, return_loader=True, train_samples=50000, test_samples=5000)
    with open(f'data/{dataset}_{setting}_paired.pkl', 'wb') as f:
        pickle.dump({'train': paired_train, 'test': paired_test}, f)
        print('Paired data saved to pickle file')

# Generate unpaired data
train, test = get_rec_data(X, dataset, setting, T, unpaired_batch_size)

In [ ]:
# Initialize the model
model = Model(num_class=120, num_point=V, num_person=M, graph='graph.ntu_rgb_d.Graph',
              graph_args={'labeling_mode': 'spatial'}, debug=False)
model = model.cuda()

model.load_state_dict(torch.load('model.pth'))